# ELM Radiometric Calibration Using 4 Tarp Panels (DJI Mavic 3M)

This notebook performs DN-to-reflectance calibration via the Empirical Line Method (ELM),
using a single capture that contains all 4 tarp panels (10%, 20%, 30%, 40% reflectance) per
DJI M3M multispectral band (Green, Red, RedEdge, NIR).

Compared to Metashape's built-in radiometric calibration (which only supports one reflectance
target), this notebook fits a separate linear model per band using all 4 known reflectance
levels, which is expected to improve calibration accuracy.

**No imagery is included in this repository** (the source drone captures belong to the host
research institution). See `examples/` for result figures from an actual run, and
`README.md` for how to point this notebook at your own data.

**Inputs** (defaults are relative paths under `./data`, which is git-ignored; override via
environment variables)
- Tarp capture (4 single-band TIFFs): `DJI_M3M_TARP_DIR` (default `./data/tarps`)
- DN orthomosaic to calibrate: `DJI_M3M_MAIN_DN_PATH` (default `./data/sample_dn_orthomosaic.tif`)

**Outputs**
- Calibrated GeoTIFFs and validation figures are written under `DJI_M3M_OUTPUT_DIR` (default `./output/`, also git-ignored)

**Note**: DJI M3M's 4 band sensors have slightly different optical centers (confirmed via XMP
`RelativeOpticalCenterX/Y`, offsets up to ~50px at this flight altitude), so tarp ROIs are
selected independently for each band rather than reused across bands.

In [1]:
import os
import cv2
import numpy as np
import rasterio
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt

%matplotlib inline

In [ ]:
# True reflectance of the 4 tarp panels (10%, 20%, 30%, 40%)
TRUE_REFLECTANCE = np.array([0.10, 0.20, 0.30, 0.40])
TARP_PCTS = [10, 20, 30, 40]

# Band index -> name / tarp filename mapping (DJI M3M has no blue band)
BAND_CONFIG = {
    1: {"name": "Green", "filename": "tarp_green.tif"},
    2: {"name": "Red", "filename": "tarp_red.tif"},
    3: {"name": "RedEdge", "filename": "tarp_rededge.tif"},
    4: {"name": "NIR", "filename": "tarp_nir.tif"},
}

# Tarp capture image paths (single image with all 4 panels per band).
# `data/` is git-ignored (not redistributed) - populate it locally with your own capture,
# or override via environment variables for portability across machines.
tarp_dir = os.environ.get("DJI_M3M_TARP_DIR", "./data/tarps")
tarp_paths = {b_idx: os.path.join(tarp_dir, info["filename"]) for b_idx, info in BAND_CONFIG.items()}

# DN orthomosaic to calibrate (band order assumed to match BAND_CONFIG: Green, Red, RedEdge, NIR).
main_dn_path = os.environ.get("DJI_M3M_MAIN_DN_PATH", "./data/sample_dn_orthomosaic.tif")

# Two calibration variants are compared: a free-intercept fit, and a black-level-corrected
# fit forced through the origin (mirrors Metashape's single-panel CRP assumption).
output_dir = os.environ.get("DJI_M3M_OUTPUT_DIR", "./output")
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "calibrated_reflectance_free_intercept.tif")
output_path_bl = os.path.join(output_dir, "calibrated_reflectance_black_level.tif")

In [ ]:
# Verify all input paths exist before doing any manual work
for b_idx, path in tarp_paths.items():
    status = "OK" if os.path.exists(path) else "MISSING"
    print(f"[{BAND_CONFIG[b_idx]['name']}] {status} -> {path}")

print(f"[Main DN tif] {'OK' if os.path.exists(main_dn_path) else 'MISSING'} -> {main_dn_path}")

with rasterio.open(main_dn_path) as src:
    print(f"Main DN tif: {src.count} bands, {src.width}x{src.height}, dtype={src.dtypes[0]}")

## Step 1: Manually select the 4 tarp ROIs per band

Run the cell below interactively in Jupyter (not headlessly). For each band:
1. Drag a box around the rough tarp area in the overview window and press Enter to zoom in.
2. For each of the 4 tarp panels in order (10% -> 20% -> 30% -> 40%), left-click the polygon
   vertices around the tarp's actual (possibly tilted) edges, then press Enter to confirm.
   Right-click undoes the last vertex; Esc clears all vertices to restart that tarp.

A polygon (rather than an axis-aligned rectangle) is used because the tarps are not always
aligned with the image axes, so a rectangular box could include background pixels or clip
tarp corners.

This cell opens OpenCV GUI windows and requires manual mouse input, so it cannot be executed
automatically.

In [4]:
def select_polygon_roi(window_title, image):
    """Let the user click polygon vertices on `image`.

    Left-click adds a vertex, right-click removes the last vertex, Esc clears all
    vertices, Enter confirms (requires >= 3 vertices). Returns an (N, 2) int32 array
    of vertex coordinates in the image's own pixel space.
    """
    points = []
    base = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR) if image.ndim == 2 else image.copy()

    def render():
        canvas = base.copy()
        for i, p in enumerate(points):
            cv2.circle(canvas, p, 4, (0, 0, 255), -1)
            if i > 0:
                cv2.line(canvas, points[i - 1], p, (0, 255, 0), 1)
        if len(points) > 2:
            cv2.line(canvas, points[-1], points[0], (0, 255, 0), 1, cv2.LINE_AA)
        cv2.imshow(window_title, canvas)

    def on_mouse(event, x, y, flags, param):
        if event == cv2.EVENT_LBUTTONDOWN:
            points.append((x, y))
            render()
        elif event == cv2.EVENT_RBUTTONDOWN and points:
            points.pop()
            render()

    cv2.namedWindow(window_title, cv2.WINDOW_NORMAL)
    cv2.setMouseCallback(window_title, on_mouse)
    render()

    while True:
        key = cv2.waitKey(20) & 0xFF
        if key in (13, 10) and len(points) >= 3:  # Enter confirms
            break
        if key == 27:  # Esc clears and restarts this tarp
            points.clear()
            render()

    cv2.destroyWindow(window_title)
    return np.array(points, dtype=np.int32)


def mean_dn_in_polygon(band_data, polygon_pts):
    """Mean DN of pixels inside a polygon (in band_data's own pixel coordinates)."""
    mask = np.zeros(band_data.shape, dtype=np.uint8)
    cv2.fillPoly(mask, [polygon_pts], 1)
    return float(band_data[mask.astype(bool)].mean())


# Mean DN value per band per tarp panel
band_tarp_dns = {b_idx: [] for b_idx in BAND_CONFIG.keys()}
band_tarp_polygons = {b_idx: [] for b_idx in BAND_CONFIG.keys()}  # kept for sanity-check plots

for b_idx, info in BAND_CONFIG.items():
    band_name = info["name"]
    tarp_file = tarp_paths[b_idx]

    print(f"\n=== [{band_name} band] Select tarp ROIs ===")

    with rasterio.open(tarp_file) as src:
        tarp_band_data = src.read(1)

    vis_band = tarp_band_data.astype(np.float32)
    vis_band_normalized = cv2.normalize(vis_band, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

    # Step 1: rough rectangular selection on a downscaled preview, just for zooming in
    target_w = 1200
    orig_h, orig_w = vis_band_normalized.shape[:2]
    scale = target_w / orig_w
    target_h = int(orig_h * scale)
    preview_img = cv2.resize(vis_band_normalized, (target_w, target_h))

    win_title_1 = f"[{band_name}] STEP 1: Drag around tarp area, then press Enter"
    cv2.namedWindow(win_title_1, cv2.WINDOW_NORMAL)
    broad_roi = cv2.selectROI(win_title_1, preview_img, fromCenter=False, showCrosshair=False)
    cv2.destroyWindow(win_title_1)

    x_p, y_p, w_p, h_p = broad_roi
    x_orig, y_orig = max(0, int(x_p / scale)), max(0, int(y_p / scale))
    w_orig, h_orig = min(orig_w - x_orig, int(w_p / scale)), min(orig_h - y_orig, int(h_p / scale))

    # Step 2: select each of the 4 tarp panels as a polygon in the zoomed-in crop
    crop_img = vis_band_normalized[y_orig:y_orig + h_orig, x_orig:x_orig + w_orig]

    for pct in TARP_PCTS:
        win_title_2 = (
            f"[{band_name}] {pct}% tarp: left-click vertices, right-click undo, "
            f"Enter confirm (>=3 pts)"
        )
        poly_local = select_polygon_roi(win_title_2, crop_img)
        poly_global = poly_local + np.array([x_orig, y_orig])

        mean_dn = mean_dn_in_polygon(tarp_band_data, poly_global)
        band_tarp_dns[b_idx].append(mean_dn)
        band_tarp_polygons[b_idx].append(poly_global)
        print(f"  {pct}% tarp -> mean DN: {mean_dn:.2f} ({len(poly_local)} vertices)")

    cv2.destroyAllWindows()

print("\nAll tarp DN values extracted:")
for b_idx, info in BAND_CONFIG.items():
    print(f"  [{info['name']}] {[f'{v:.2f}' for v in band_tarp_dns[b_idx]]}")


=== [Green band] Select tarp ROIs ===


c:\Users\Seongjun\miniconda3\envs\lsj\Lib\site-packages\rasterio\__init__.py:379: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)


  10% tarp -> mean DN: 21849.57 (4 vertices)
  20% tarp -> mean DN: 41667.21 (4 vertices)
  30% tarp -> mean DN: 50761.26 (4 vertices)
  40% tarp -> mean DN: 64788.28 (4 vertices)

=== [Red band] Select tarp ROIs ===
  10% tarp -> mean DN: 15656.14 (4 vertices)
  20% tarp -> mean DN: 30258.92 (4 vertices)
  30% tarp -> mean DN: 37437.69 (4 vertices)
  40% tarp -> mean DN: 51044.71 (4 vertices)

=== [RedEdge band] Select tarp ROIs ===
  10% tarp -> mean DN: 12013.46 (4 vertices)
  20% tarp -> mean DN: 21706.78 (4 vertices)
  30% tarp -> mean DN: 26918.30 (4 vertices)
  40% tarp -> mean DN: 36932.15 (4 vertices)

=== [NIR band] Select tarp ROIs ===
  10% tarp -> mean DN: 10215.34 (4 vertices)
  20% tarp -> mean DN: 16383.24 (4 vertices)
  30% tarp -> mean DN: 20153.31 (4 vertices)
  40% tarp -> mean DN: 27402.54 (4 vertices)

All tarp DN values extracted:
  [Green] ['21849.57', '41667.21', '50761.26', '64788.28']
  [Red] ['15656.14', '30258.92', '37437.69', '51044.71']
  [RedEdge] ['1201

In [ ]:
# Fit a linear ELM model (Reflectance = m * DN + c) per band
calibration_models = {}
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for b_idx, info in BAND_CONFIG.items():
    band_name = info["name"]
    X = np.array(band_tarp_dns[b_idx]).reshape(-1, 1)
    y = TRUE_REFLECTANCE

    model = LinearRegression().fit(X, y)
    m, c = model.coef_[0], model.intercept_
    calibration_models[b_idx] = (m, c)
    r2 = model.score(X, y)
    print(f"[{band_name}] R2={r2:.4f}  Reflectance = {m:.6e} * DN + ({c:.4f})")

    ax = axes.flat[b_idx - 1]
    ax.scatter(X, y, color="darkblue", label="Tarp samples")
    ax.plot(X, model.predict(X), color="orange", label=f"R2={r2:.3f}")
    ax.set_title(f"{band_name} band ELM fit")
    ax.set_xlabel("DN")
    ax.set_ylabel("Reflectance")
    ax.tick_params(axis="both", labelsize=12)
    ax.legend()

fig.tight_layout()
fig.savefig(os.path.join(output_dir, "elm_fit.png"), dpi=300)
plt.show()

In [ ]:
# Alternative model: black-level-corrected, origin-forced fit
# (Reflectance = m * (DN - black_level)), i.e. DN == black_level -> Reflectance == 0.
#
# This mirrors Metashape's single-panel CRP assumption (DN=0 -> Reflectance=0), but first
# removes the sensor's documented black level offset (XMP `BlackLevel`) rather than forcing
# the line through DN=0 directly. Compare against the free-intercept model fit above.
from PIL import Image

calibration_models_bl = {}

print(f"{'Band':<10}{'BlackLevel':<12}{'R2 (free intercept)':<22}{'R2 (black-level, forced 0)'}")
for b_idx, info in BAND_CONFIG.items():
    band_name = info["name"]
    xmp = Image.open(tarp_paths[b_idx]).getxmp()["xmpmeta"]["RDF"]["Description"]
    black_level = float(xmp["BlackLevel"])

    X = np.array(band_tarp_dns[b_idx])
    y = TRUE_REFLECTANCE

    m_bl = np.sum((X - black_level) * y) / np.sum((X - black_level) ** 2)
    c_bl = -m_bl * black_level
    calibration_models_bl[b_idx] = (m_bl, c_bl)

    m_free, c_free = calibration_models[b_idx]
    pred_free = m_free * X + c_free
    pred_bl = m_bl * X + c_bl
    r2_free = 1 - np.sum((y - pred_free) ** 2) / np.sum((y - y.mean()) ** 2)
    r2_bl = 1 - np.sum((y - pred_bl) ** 2) / np.sum((y - y.mean()) ** 2)
    print(f"{band_name:<10}{black_level:<12.0f}{r2_free:<22.4f}{r2_bl:.4f}")

# Visualize both models, extrapolated down toward DN=0, to see where they diverge
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for b_idx, info in BAND_CONFIG.items():
    band_name = info["name"]
    X = np.array(band_tarp_dns[b_idx])
    y = TRUE_REFLECTANCE
    m_free, c_free = calibration_models[b_idx]
    m_bl, c_bl = calibration_models_bl[b_idx]

    dn_range = np.linspace(0, X.max() * 1.05, 200)
    ax = axes.flat[b_idx - 1]
    ax.scatter(X, y, color="black", zorder=5, label="Tarp samples")
    ax.plot(dn_range, m_free * dn_range + c_free, color="orange", label="Free intercept")
    ax.plot(dn_range, m_bl * dn_range + c_bl, color="green", label="Black-level corrected (forced 0)")
    ax.axhline(0, color="gray", linestyle=":", linewidth=1)
    ax.set_title(f"{band_name} band: model comparison")
    ax.set_xlabel("DN")
    ax.set_ylabel("Reflectance")
    ax.tick_params(axis="both", labelsize=12)
    ax.legend(fontsize=8)

fig.tight_layout()
fig.savefig(os.path.join(output_dir, "elm_model_comparison.png"), dpi=300)
plt.show()

In [ ]:
# Apply both calibration models (free-intercept vs black-level-corrected) to the full DN orthomosaic
OUTPUT_NODATA = -9999.0  # cannot collide with the valid reflectance range [0, 1]

with rasterio.open(main_dn_path) as main_src:
    main_meta = main_src.meta.copy()
    main_img_array = main_src.read()  # shape: [bands, height, width]
    src_nodata = main_src.nodata

# A pixel is nodata if ANY band is nodata there (multiband orthomosaics share one footprint)
nodata_mask = np.any(main_img_array == src_nodata, axis=0) if src_nodata is not None else np.zeros(main_img_array.shape[1:], dtype=bool)
n_valid = (~nodata_mask).sum()
print(f"Source nodata value: {src_nodata}, nodata pixels: {nodata_mask.sum()} / {nodata_mask.size}")

main_meta.update({"dtype": "float32", "count": len(BAND_CONFIG), "nodata": OUTPUT_NODATA})

models_to_apply = {
    "free_intercept": (calibration_models, output_path),
    "black_level": (calibration_models_bl, output_path_bl),
}

for label, (models, out_path) in models_to_apply.items():
    calibrated_img = np.full(main_img_array.shape, OUTPUT_NODATA, dtype=np.float32)
    for b_idx, info in BAND_CONFIG.items():
        m, c = models[b_idx]
        calibrated_band = np.clip(main_img_array[b_idx - 1] * m + c, 0.0, 1.0)
        n_zero_clipped = np.sum((calibrated_band == 0.0) & ~nodata_mask)
        calibrated_band[nodata_mask] = OUTPUT_NODATA
        calibrated_img[b_idx - 1] = calibrated_band
        print(f"  [{label}/{info['name']}] zero-clipped: {n_zero_clipped} / {n_valid} valid px ({100 * n_zero_clipped / n_valid:.1f}%)")

    with rasterio.open(out_path, "w", **main_meta) as dst:
        dst.write(calibrated_img)
    print(f"Saved [{label}] -> {out_path}\n")

In [ ]:
# Validate both calibration models: compare known tarp reflectance against calibrated values
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
true_pct = TRUE_REFLECTANCE * 100

for b_idx, info in BAND_CONFIG.items():
    raw_dns = np.array(band_tarp_dns[b_idx])
    ax = axes.flat[b_idx - 1]
    ax.plot([0, 50], [0, 50], color="gray", linestyle="--", alpha=0.7, label="Ideal 1:1")

    for label, models, color in [
        ("Free intercept", calibration_models, "tab:orange"),
        ("Black-level corrected", calibration_models_bl, "tab:green"),
    ]:
        m, c = models[b_idx]
        cal_pct = np.clip(raw_dns * m + c, 0.0, 1.0) * 100
        rmse = np.sqrt(np.mean((true_pct - cal_pct) ** 2))
        ax.scatter(true_pct, cal_pct, color=color, s=80, edgecolors="black",
                   label=f"{label} (RMSE {rmse:.2f}%)")

    ax.set_title(f"{info['name']} band validation")
    ax.set_xlabel("Target reflectance (%)")
    ax.set_ylabel("Calibrated reflectance (%)")
    ax.set_xlim(0, 50)
    ax.set_ylim(0, 50)
    ax.tick_params(axis="both", labelsize=12)
    ax.legend(loc="upper left", fontsize=8)

fig.tight_layout()
fig.savefig(os.path.join(output_dir, "elm_validation.png"), dpi=300)
plt.show()